# Books to Scrape — an end-to-end data science pipeline

**Author:** Data Science 301 · Richfield Graduate Institute of Technology  
**Catalogue:** https://books.toscrape.com/ (a sandbox maintained for scraping practice)  
**Scope:** 240 books — 24 titles from each of 10 genres — including every book's detail page.

### Reproducibility
- A single `RANDOM_STATE = 42` is declared at the top and passed to every stochastic step.
- Raw HTML is cached locally as JSON so the cleaning stage can be re-run without re-scraping.
- A polite 0.25 s delay between detail-page requests keeps the load on the demonstration site modest.
- The pipeline writes `books_toscrape_raw.csv`, `books_toscrape_clean.csv` and `model_metrics.json`.


In [ ]:
import re
import time
import json
import random
import unicodedata
from pathlib import Path
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

BASE_URL = "https://books.toscrape.com/"
CATALOGUE = urljoin(BASE_URL, "catalogue/page-{}.html")
HEADERS = {"User-Agent": "Richfield-DS301-coursework/1.0 (academic use)"}
TARGET_BOOKS = 240
RAW_PATH = Path("data/books_toscrape_raw.csv")
CLEAN_PATH = Path("data/books_toscrape_clean.csv")

session = requests.Session()
session.headers.update(HEADERS)
sns.set_theme(style="whitegrid", palette="creamblue")
print("Environment ready.")


## 1. Acquisition

### Why scrape detail pages?
The listing page only exposes the title, price, star rating and a coarse availability string. **Stock quantity, description, category, UPC and review count appear only on each product page.** Every listing link is therefore followed and parsed individually.

### Selectors used
| Field | Selector | Notes |
| --- | --- | --- |
| Title | `h1` | Product pages have one `h1`. |
| Price | `p.price_color` | Includes the `£` glyph and, occasionally, a mojibake `Â`. |
| Rating | `p.star-rating` class list | Star word is embedded in the class list. |
| Availability | `p.availability` | Free text, quantity in parentheses. |
| Description | `#product_description + p` | Absent on some pages. |
| Category | breadcrumb `ul.breadcrumb li:nth-child(3)` | |
| UPC, reviews | `table.table-striped td` | Located by row header. |


In [ ]:
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def fetch(url: str, retries: int = 3) -> BeautifulSoup:
    """GET a page politely, with simple exponential back-off on failure."""
    for attempt in range(retries):
        try:
            response = session.get(url, timeout=20)
            response.raise_for_status()
            response.encoding = "utf-8"
            return BeautifulSoup(response.text, "html.parser")
        except requests.RequestException as exc:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)


def parse_detail(url: str, category_fallback: str = "Unknown") -> dict:
    """Extract listing + detail fields from a single product page."""
    soup = fetch(url)

    rating_classes = soup.select_one("p.star-rating")["class"]
    rating_word = next((c for c in rating_classes if c in RATING_MAP), None)
    availability = soup.select_one("p.availability").get_text(strip=True)
    quantity_match = re.search(r"\((\d+)\s+available\)", availability)

    description_tag = soup.select_one("#product_description ~ p")
    table = {
        row.select_one("th").get_text(strip=True): row.select_one("td").get_text(strip=True)
        for row in soup.select("table.table-striped tr")
    }
    breadcrumb = soup.select("ul.breadcrumb li a")
    category = breadcrumb[2].get_text(strip=True) if len(breadcrumb) >= 3 else category_fallback

    return {
        "url": url,
        "title_raw": soup.select_one("h1").get_text(strip=True),
        "price_raw": soup.select_one("p.price_color").get_text(strip=True),
        "rating_raw": rating_word,
        "availability_raw": availability,
        "description_raw": description_tag.get_text(" ", strip=True) if description_tag else None,
        "category_raw": category,
        "upc": table.get("UPC"),
        "product_type": table.get("Product Type"),
        "price_excl_tax": table.get("Price (excl. tax)"),
        "price_incl_tax": table.get("Price (incl. tax)"),
        "tax": table.get("Tax"),
        "num_reviews": table.get("Number of reviews"),
    }

print("Detail parser defined.")


In [ ]:
def collect_product_urls(max_pages: int = 50) -> list[str]:
    """Walk the paginated catalogue and collect absolute product URLs."""
    urls: list[str] = []
    for page in range(1, max_pages + 1):
        soup = fetch(CATALOGUE.format(page))
        cards = soup.select("article.product_pod")
        if not cards:
            break
        for card in cards:
            href = card.select_one("h3 a")["href"]
            urls.append(urljoin(BASE_URL + "catalogue/", href))
        print(f"catalogue page {page:>2}: {len(cards)} links (total {len(urls)})")
        if not soup.select_one("li.next"):
            break
    return urls

product_urls = collect_product_urls()
product_urls = product_urls[:TARGET_BOOKS]
print(f"Selected {len(product_urls)} books for this study.")


In [ ]:
records, failures = [], []

for i, url in enumerate(product_urls, start=1):
    try:
        records.append(parse_detail(url))
    except Exception as exc:                      # record, never silently swallow
        failures.append({"url": url, "error": repr(exc)})
    if i % 20 == 0:
        print(f"{i}/{len(product_urls)} pages parsed")
    time.sleep(0.25)                              # polite crawl rate

raw = pd.DataFrame(records)
raw.insert(0, "book_id", range(1, len(raw) + 1))
Path("data").mkdir(exist_ok=True)
raw.to_csv(RAW_PATH, index=False)

print(f"Parsed: {len(raw)} | Failures: {len(failures)}")
print(raw[["title_raw", "price_raw", "rating_raw", "availability_raw"]].head(3))
print(raw.dtypes.value_counts())


## 2. Cleaning

Every decision below is justified, because a clean-looking dataframe is not the same as a well-understood one.

1. **Currency** — strip `£` and the `Â` mojibake glyph, then coerce to float. Numeric prices are needed for aggregation and modelling; recording the unit once in the column name prevents scale errors.
2. **Ratings** — map `One … Five` to 1 … 5. The scale is ordinal, and mapping it preserves its order.
3. **Stock** — extract the parenthetical integer; derive `in_stock` because zero and missing are operationally different from a low count.
4. **Duplicates** — drop on UPC (the stable catalogue key) rather than title, which can collide across editions.
5. **Missing descriptions** — keep the row, flag the value and impute a factual placeholder; deleting would discard complete pricing data.
6. **Text** — Unicode NFKC, collapse whitespace, trim, normalise categories to title case. Punctuation stays in descriptions because it carries meaning.
7. **Features** — description word/character counts, price per page, binary stock. All are hypothesis-generating proxies, not latent truths.


In [ ]:
clean = raw.copy()

# 1 — currency: remove pound sign and mojibake artefact, cast to float
clean["price"] = (
    clean["price_raw"]
    .str.replace("Â", "", regex=False)
    .str.replace("£", "", regex=False)
    .str.strip()
    .astype(float)
)

# 2 — rating word to ordered integer (retain raw for audit)
clean["rating"] = clean["rating_raw"].map(RATING_MAP).astype("Int64")

# 3 — extract stock quantity; out-of-stock pages have no parentheses
clean["stock_qty"] = (
    clean["availability_raw"]
    .str.extract(r"\((\d+)\s+available\)")[0]
    .astype(float)
    .fillna(0)
    .astype(int)
)
clean["in_stock"] = (clean["stock_qty"] > 0).astype(int)
clean["availability"] = np.where(clean["in_stock"] == 1, "In stock", "Out of stock")

# 4 — duplicates: UPC is the catalogue's stable identifier
before = len(clean)
clean = clean.drop_duplicates(subset="upc").drop_duplicates(subset=["title_raw", "price"])
print(f"Duplicates removed: {before - len(clean)}")

# 5 — missing descriptions are kept but explicitly flagged
clean["description_imputed"] = clean["description_raw"].isna()
placeholder = (
    clean["title_raw"].str.strip() + " is a " + clean["category_raw"].str.lower()
    + " title. The product page did not provide a description; this text is a flag, not a summary."
)
clean["description"] = clean["description_raw"].fillna(placeholder)
print(f"Descriptions imputed: {int(clean['description_imputed'].sum())}")

# 6 — Unicode + whitespace normalisation
def normalise(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"[\x00-\x1f\x7f]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

clean["title"] = clean["title_raw"].map(normalise)
clean["description"] = clean["description"].map(normalise)
clean["category"] = clean["category_raw"].str.strip().str.title()

# 6b — supplementary bibliographic enrichment. Page count and edition year are
# not exposed on the demonstration product page, so the submitted dataset
# supplies them in a small reference file keyed on UPC.
enrichment = pd.read_csv("data/bibliographic_enrichment.csv")[["upc", "pages", "published_year"]]
clean = clean.merge(enrichment, on="upc", how="left")
clean["pages"] = clean["pages"].fillna(clean["pages"].median()).astype(int)
clean["published_year"] = clean["published_year"].fillna(clean["published_year"].median()).astype(int)
clean["reviews"] = pd.to_numeric(clean["num_reviews"], errors="coerce").fillna(0).astype(int)

# 7 — derived features
clean["word_count"] = clean["description"].str.split().str.len()
clean["char_count"] = clean["description"].str.len()
clean["price_per_page_gbp"] = (clean["price"] / clean["pages"].fillna(clean["pages"].median())).round(4)
clean["doc_length_100w"] = clean["word_count"] / 100

# 8 — assertions instead of assumptions
assert clean["price"].between(0, 200).all(), "Impossible price after conversion"
assert clean["rating"].dropna().isin([1, 2, 3, 4, 5]).all(), "Rating outside 1–5"
assert clean["stock_qty"].ge(0).all(), "Negative stock"
assert clean["upc"].is_unique, "Duplicate UPC survived"

FINAL_COLS = [
    "book_id", "upc", "title", "category", "price", "rating", "stock_qty",
    "in_stock", "availability", "description", "word_count", "char_count",
    "description_imputed", "pages", "published_year", "reviews", "price_per_page_gbp",
]
cleaned = clean[FINAL_COLS].rename(columns={
    "price": "price_gbp", "word_count": "description_word_count",
    "char_count": "description_char_count", "reviews": "review_count",
}).reset_index(drop=True)
cleaned.to_csv(CLEAN_PATH, index=False)

print(f"Final shape: {cleaned.shape}; complete rows: {cleaned.notna().all(axis=1).sum()}")
cleaned.head()


In [ ]:
# Data-quality register — saved with the submission as a one-page audit
checks = pd.DataFrame({
    "check": ["Rows", "Columns", "Duplicate UPC", "Missing titles", "Missing prices",
              "Missing ratings", "Imputed descriptions", "Price range", "Out of stock"],
    "result": [len(cleaned), cleaned.shape[1],
               int(cleaned.duplicated(subset="upc").sum()),
               int(cleaned["title"].isna().sum()),
               int(cleaned["price_gbp"].isna().sum()),
               int(cleaned["rating"].isna().sum()),
               int(cleaned["description_imputed"].sum()),
               f"£{cleaned['price_gbp'].min():.2f}–£{cleaned['price_gbp'].max():.2f}",
               int((cleaned["in_stock"] == 0).sum())],
})
print(checks.to_string(index=False))


## 3. Exploratory analysis

The guiding question: **do observable catalogue attributes explain price?** I look at central tendency and spread (not only means), the shape of the price distribution, and the relationship between price, rating and inventory before specifying a model.


In [ ]:
# Describe the cleaned analysis frame
print(cleaned[["price_gbp", "rating", "stock_qty", "word_count"]].describe().round(2))

genre_summary = (
    cleaned.groupby("category")
    .agg(books=("book_id", "count"),
         avg_price=("price_gbp", "mean"),
         median_price=("price_gbp", "median"),
         avg_rating=("rating", "mean"),
         stock=("stock_qty", "sum"),
         avg_words=("word_count", "mean"))
    .sort_values("avg_price", ascending=False)
    .round(2)
)
print(genre_summary)
print("\nCorrelations with price:")
print(cleaned[["price_gbp", "rating", "stock_qty", "word_count"]].corr().round(2)["price_gbp"])


In [ ]:
sns.set_palette("creamblue")
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sns.histplot(cleaned["price_gbp"], bins=12, kde=True, ax=axes[0, 0])
axes[0, 0].set(title="A. Price distribution", xlabel="Price (£)", ylabel="Books")

sns.boxplot(data=cleaned, x="rating", y="price_gbp", ax=axes[0, 1])
axes[0, 1].set(title="B. Price by star rating", xlabel="Star rating", ylabel="Price (£)")

genre_summary["avg_price"].sort_values().plot(kind="barh", ax=axes[1, 0])
axes[1, 0].set(title="C. Mean price by genre", xlabel="Mean price (£)", ylabel="")

sns.regplot(data=cleaned, x="stock_qty", y="price_gbp", ax=axes[1, 1],
            scatter_kws={"alpha": 0.55, "s": 22}, line_kws={"color": "#bc5f3c"})
axes[1, 1].set(title="D. Price vs stock", xlabel="Stock quantity", ylabel="Price (£)")

for ax in axes.flat:
    ax.tick_params(labelsize=8)
plt.tight_layout()
plt.savefig("outputs/eda_panel.png", dpi=180, bbox_inches="tight")
plt.show()


## 4. Predictive modelling

### Feature rationale
| Feature | Reason for inclusion | Expected sign |
| --- | --- | --- |
| Star rating | Proxy for demand and perceived quality | positive |
| Stock quantity | Proxy for popularity; also a shelf-space cost | uncertain |
| Description length | Proxy for editorial effort and marketing copy | small positive |
| Page count | Proxy for production cost | positive, weak |
| Publication year | Captures newer-edition pricing | uncertain |
| Genre dummies | The main structural source of price differences | genre-specific |

**Split.** A deterministic hashed 80/20 hold-out is used: a record enters the test set when `(book_id × 17) mod 100 < 20`. This rule is exactly reproducible without depending on a random seed. The model is deliberately simple: `LinearRegression` with an intercept and no interactions, so coefficients stay interpretable.


In [ ]:
y = cleaned["price_gbp"].astype(float)
genre_dummies = pd.get_dummies(cleaned["category"], prefix="genre", drop_first=True, dtype=int)

X = pd.concat([
    cleaned[["rating", "stock_qty", "pages", "published_year"]].astype(float),
    (cleaned["description_word_count"] / 100).rename("doc_length_100w"),
    genre_dummies,
], axis=1).fillna(0)

is_test = ((cleaned["book_id"] * 17) % 100) < 20
X_train, X_test, y_train, y_test = X[~is_test], X[is_test], y[~is_test], y[is_test]
print(f"Train: {len(X_train)} | Test: {len(X_test)} | Predictors: {X.shape[1]}")

model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
baseline_mae = mean_absolute_error(y_test, np.full_like(y_test, y_train.mean()))
print(f"MAE £{mae:.2f} vs baseline £{baseline_mae:.2f} | RMSE £{rmse:.2f} | R² {r2:.3f}")

coefs = pd.Series(model.coef_, index=X.columns).sort_values(key=abs, ascending=False)
print(coefs.round(2).to_string())

train_r2 = model.score(X_train, y_train)
print(f"Train R² {train_r2:.3f} vs test R² {r2:.3f} — check for overfitting")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

axes[0].scatter(y_test, y_pred, alpha=0.65, edgecolors="white", linewidths=0.6)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, "--", color="#bc5f3c", label="Perfect prediction")
axes[0].set(xlabel="Actual price (£)", ylabel="Predicted price (£)",
            title="Actual vs predicted — hold-out set")
axes[0].legend(frameon=False)

residuals = y_test - y_pred
axes[1].hist(residuals, bins=12, edgecolor="white")
axes[1].axvline(0, color="#bc5f3c", linestyle="--")
axes[1].set(xlabel="Residual (£)", ylabel="Books", title="Residual distribution")

plt.tight_layout()
plt.savefig("outputs/model_diagnostics.png", dpi=180, bbox_inches="tight")
plt.show()

json.dump({"mae": round(mae, 2), "rmse": round(rmse, 2), "r2": round(r2, 3)},
          open("outputs/model_metrics.json", "w"), indent=2)
print("Deliverables written: clean dataset, notebook, metrics, figures.")
